# Split

The same time split as the complaint-escalation project, so the two can be compared: train on narratives received up to 30 Sep 2016 and test on 1 Oct 2016 to 28 Feb 2017. The test set isn't used for anything until the final model is picked.

In [1]:
import sys

sys.path.append("..")

import pandas as pd
from src.data import load_text, time_split

In [2]:
df = load_text()
train, test = time_split(df)
len(train), len(test)

(112256, 37405)

In [3]:
summary = pd.DataFrame({
    "rows": [len(train), len(test)],
    "disputed": [train["disputed"].sum(), test["disputed"].sum()],
    "dispute_rate": [train["disputed"].mean(), test["disputed"].mean()],
    "first": [train["date_received"].min(), test["date_received"].min()],
    "last": [train["date_received"].max(), test["date_received"].max()],
}, index=["train", "test"])
summary["share"] = summary["rows"] / len(df)
summary.round({"dispute_rate": 3, "share": 3})

,rows,disputed,dispute_rate,first,last,share
train,112256,26089,0.232,2015-03-19,2016-09-30,0.75
test,37405,7795,0.208,2016-10-01,2017-02-28,0.25


## What changes between train and test

In [4]:
pd.concat([train["product"].value_counts(normalize=True), test["product"].value_counts(normalize=True)],
          axis=1, keys=["train", "test"]).round(3)

,train,test
product,,
Debt collection,0.254,0.22
Mortgage,0.215,0.176
Credit reporting,0.19,0.186
Credit card,0.117,0.113
Bank account or service,0.09,0.097
Consumer Loan,0.057,0.059
Student loan,0.044,0.123
Payday loan,0.011,0.01
Prepaid card,0.01,0.006


In [5]:
new_companies = set(test["company"]) - set(train["company"])
len(new_companies), test["company"].isin(new_companies).sum()

(277, np.int64(534))

In [6]:
train["text"].str.split().str.len().median(), test["text"].str.split().str.len().median()

(np.float64(137.0), np.float64(133.0))

The dispute rate drops from 23.2% in train to 20.8% in test, the same direction as in the first project. Student loans go from 4.4% of narratives to 12.3% and mortgages from 21.5% to 17.6%. 277 companies (534 complaints) only show up in the test window. The narratives themselves are about as long as before, with a median of 133 words against 137.

## Texts on both sides

In [7]:
train_text = train["text"].str.lower()
test_text = test["text"].str.lower()
exact = test_text.isin(train_text)
exact.sum(), test.loc[exact, "disputed"].mean()

(np.int64(77), np.float64(0.06493506493506493))

In [8]:
same_opening = test_text.str[:100].isin(train_text.str[:100])
same_opening.sum(), same_opening.mean()

(np.int64(286), np.float64(0.007646036626119503))

In [9]:
test_text.str[:100][same_opening].value_counts().head(5)

text
i was shocked when i reviewed my credit report and found late payment on the dates below : 30 days p    29
transunion had a glitch in their system that violated the rights of their consumers under section 6.    22
i am filing this complaint because equifax has ignored my request to open investigation in regards t    22
i 've sent multiple letters to this agency about this not being my account. after being advised by i    12
i am filling this complaint because equifax has ignored my request to provide me with the documents      7
Name: count, dtype: int64

In [10]:
test.groupby(same_opening.rename("opening_in_train"))["disputed"].agg(["size", "mean"]).round(3)

,size,mean
opening_in_train,,
False,37119,0.209
True,286,0.143


77 test narratives are exact copies of a training text, and 286 (0.8%) start with the same 100 characters as one. They're the template letters from the EDA, and they're disputed less than the rest of the test set (14.3% against 20.9%). They stay in. A model in use would also get letters it has seen before, and 286 complaints out of 37,405 can't move the test score much. The final test run will give the score without them as well.